# CLIP-Guided StyleGAN2 with 5 Learned Mapping Networks

## Overview

This notebook implements a pipeline where **5 separate mapping networks** (initialised from the pretrained StyleGAN2 mapping network) learn to convert CLIP image embeddings into the W+ latent space of a **StyleGAN2 synthesis network** to generate **256×256** face images.

### Architecture

```
Real Image ──► CLIP ViT-B/32 ──► 512-dim embedding ──► 5 Mapping Networks ──► W+ (14 × 512) ──► Synthesis Network ──► Generated Image
                                                                                                                           │
                                                                   ┌───────────────────────────────────────────────────────┘
                                                                   │
                                                                   ├──► Discriminator ──► Adversarial Loss
                                                                   │
                                                                   └──► CLIP ──► Cosine Similarity with Real ──► CLIP Loss
```

### Mapper-to-Layer Assignments

For 256×256 resolution, StyleGAN2 uses **14 style layers** (num_ws=14). Each mapper produces a single 512-dim **w** vector that is broadcast to its assigned layers:

| Mapper | Resolution Block | W Layers | # of w values |
|--------|-----------------|----------|---------------|
| Mapper 0 | 4×4 | 0, 1 | 2 × 512 |
| Mapper 1 | 8×8 | 2, 3 | 2 × 512 |
| Mapper 2 | 16×16 | 4, 5 | 2 × 512 |
| Mapper 3 | 32×32 | 6, 7 | 2 × 512 |
| Mapper 4 | 64×64, 128×128, 256×256 | 8–13 | 6 × 512 |

**Total: 14 × 512 = 7,168 w values** forming the complete W+ tensor.

### Loss Function

$$\mathcal{L} = \mathcal{L}_{adv} + \lambda \cdot \mathcal{L}_{CLIP}$$

- $\mathcal{L}_{adv}$: Non-saturating generator loss from frozen Discriminator
- $\mathcal{L}_{CLIP} = 1 - \cos(\text{CLIP}(x_{real}),\; \text{CLIP}(x_{fake}))$
- $\lambda$: Adjustable weight (default 1.0)

### What is Frozen vs Trainable

| Component | Frozen? | Trainable? |
|-----------|---------|------------|
| CLIP ViT-B/32 | ✓ | ✗ |
| StyleGAN2 Synthesis Network | ✓ | ✗ |
| StyleGAN2 Discriminator | ✓ | ✗ |
| 5 Mapping Networks | ✗ | ✓ |

## Step 1 – Install Dependencies & Clone StyleGAN2 Repository

Install PyTorch, torchvision, CLIP dependencies, and clone the StyleGAN2-ADA-PyTorch repository. We use a fork with updated dependencies for compatibility with recent PyTorch versions. The CLIP model (ViT-B/32) is also installed from OpenAI's repository.

In [1]:
!pip install torch torchvision ftfy regex tqdm
!pip install ninja matplotlib pillow
!pip install wandb
!pip install tqdm

In [2]:
!rm -rf stylegan2-ada-pytorch
!git clone https://github.com/RashmikaDushan/stylegan2-ada-pytorch.git

Cloning into 'stylegan2-ada-pytorch'...
remote: Enumerating objects: 256, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 256 (delta 57), reused 60 (delta 30), pack-reused 158 (from 3)
Receiving objects: 100% (256/256), 1.40 MiB | 6.04 MiB/s, done.
Resolving deltas: 100% (124/124), done.


In [3]:
%cd stylegan2-ada-pytorch
!git checkout dependency-updates

/home/m2s/mind2sketch/Mapping-Neural-Networks/stylegan2-ada-pytorch
branch 'dependency-updates' set up to track 'origin/dependency-updates'.
Switched to a new branch 'dependency-updates'


In [4]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-nct2td0n
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-nct2td0n
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Step 2 – Mount Google Drive & Import Libraries

Mount Google Drive to access the pretrained StyleGAN2 checkpoint (`.pkl` file) and the image dataset stored there. Then import all required Python libraries including PyTorch, CLIP, and the StyleGAN2-ADA modules (`dnnlib`, `legacy`).

In [5]:
!pip install click

In [ ]:
import os
import sys
import copy
import math
import torch
import clip
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, utils
from PIL import Image

In [7]:
sys.path.append('./stylegan2-ada-pytorch')

import dnnlib
import legacy

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


## Step 3 – Load CLIP Model (ViT-B/32) and Set Device

Set the compute device to GPU (`cuda`) if available, otherwise CPU.

Load the **CLIP ViT-B/32** model which produces **512-dimensional** image embeddings. The CLIP model is **frozen** — its parameters are not updated during training. It serves two purposes:
1. **Encode real images** from the dataset into 512-dim embeddings (input to the mapping networks)
2. **Encode generated images** to compute the CLIP similarity loss against real image embeddings

In [9]:
clip_model, _ = clip.load("ViT-B/32", device=device)
clip_model.eval()

for p in clip_model.parameters():
    p.requires_grad = False

## Step 4 – Configuration & Hyperparameters

Set the paths and training hyperparameters:
- **`CHECKPOINT_PATH`**: Path to the pretrained StyleGAN2 `.pkl` file on Google Drive (trained on FFHQ at 256×256)
- **`FFHQ_DIR`**: Path to the face image dataset used as real images
- **`BATCH_SIZE`**: Number of images per training step
- **`LR`**: Learning rate for Adam optimizer (only applied to the 5 mapping networks)
- **`LAMBDA_CLIP`**: Weight for the CLIP similarity loss term in the total loss: $\mathcal{L} = \mathcal{L}_{adv} + \lambda \cdot \mathcal{L}_{CLIP}$
- **`SAVE_DIR`**: Directory to save mapper checkpoints and sample images

In [ ]:
CHECKPOINT_PATH = '/home/m2s/mind2sketch/Mapping-Neural-Networks/weights/pretrained/stylegan2-ffhq-256x256.pkl'
FFHQ_DIR = '/home/m2s/mind2sketch/Mapping-Neural-Networks/ffhq/ffhq256'
BATCH_SIZE = 16
LR = 2e-3
EPOCHS = 100
LAMBDA_CLIP = 1
VAL_SPLIT = 0.1
TEST_SPLIT = 0.1
SAVE_DIR = '/home/m2s/mind2sketch/Mapping-Neural-Networks/weights/5_mapper_b16'

os.makedirs(FFHQ_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)


## Step 5 – Load Pretrained StyleGAN2 (Generator & Discriminator)

Load the pretrained StyleGAN2 model from the `.pkl` checkpoint on Google Drive. This gives us:
- **`G` (Generator – G_ema)**: Contains both the **mapping network** and the **synthesis network**. We will deep-copy the mapping network 5 times to create our 5 mappers, and use the synthesis network (frozen) to generate images.
- **`D` (Discriminator)**: Used to compute the adversarial loss. Also **frozen** during training.

Both G and D are set to `eval()` mode and all their parameters have `requires_grad = False` — they are not updated during training. Only the 5 mapping networks we create will be trainable.

In [11]:
print('Loading StyleGAN2 checkpoint (must be a .pkl from stylegan2-ada)...')

if not os.path.exists(CHECKPOINT_PATH):
    print(f'WARNING: checkpoint not found at {CHECKPOINT_PATH}. Upload the checkpoint to Colab or change the path.')
    G = None
    D = None
else:
    with open(CHECKPOINT_PATH, 'rb') as f:
        print('Opening checkpoint...')
        model_data = legacy.load_network_pkl(f)

    G = model_data['G_ema'].to(device)
    D = model_data['D'].to(device)

    print('Loaded G and D from checkpoint.')

Loading StyleGAN2 checkpoint (must be a .pkl from stylegan2-ada)...
Opening checkpoint...
Loaded G and D from checkpoint.


In [12]:
if G is not None:
    for p in G.parameters():
        p.requires_grad = False
    G.eval()

if D is not None:
    for p in D.parameters():
        p.requires_grad = False
    D.eval()

## Step 6 – CLIP Encoding Function

Define a function to encode image tensors through the frozen CLIP model:

1. Convert images from `[-1, 1]` range (StyleGAN2 output format) to `[0, 1]`
2. Resize to 224×224 (CLIP's expected input size)
3. Apply CLIP's ImageNet normalization
4. Pass through `clip_model.encode_image()` to get 512-dim embeddings
5. **Cast from float16 to float32** — CLIP on CUDA outputs half-precision, but our mapping networks operate in float32. This cast fixes the `RuntimeError: mat1 and mat2 must have the same dtype` error.
6. L2-normalize the embeddings

When `allow_grad=True`, gradients flow through the CLIP encoder (needed for the generated-image path so that the CLIP loss backpropagates to the mapping networks).

In [13]:
clip_input_size = 224

def encode_with_clip_from_tensor(img_tensor, allow_grad=False):
    """
    img_tensor: (B, 3, H, W) in [-1, 1]
    Returns: (B, 512) float32 CLIP embedding, L2-normalized
    """
    x = (img_tensor + 1.0) / 2.0  # to [0, 1]
    x = F.interpolate(x, size=(clip_input_size, clip_input_size), mode='bilinear', align_corners=False)

    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=x.device).view(1, 3, 1, 1)
    std  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=x.device).view(1, 3, 1, 1)
    x = (x - mean) / std

    if allow_grad:
        emb = clip_model.encode_image(x)
    else:
        with torch.no_grad():
            emb = clip_model.encode_image(x)

    # CLIP on CUDA outputs float16 — cast to float32 to avoid dtype mismatch
    emb = emb.float()
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb

## Step 7 – Image Preprocessing Transform

Define the transform pipeline for loading real images from the dataset:
1. Resize to 256×256 (matching StyleGAN2's output resolution)
2. Center crop to handle non-square images
3. Convert to tensor (`[0, 1]` range)
4. Normalize to `[-1, 1]` range (standard for StyleGAN2 and GAN training)

In [14]:
real_image_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),  # gives [0,1]
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]) # -> [-1,1]
])

## Step 8 – Define the Five Mapping Networks (`FiveMappers`)

This is the core trainable component. We create **5 mapping networks**, each initialised as a **deep copy of the pretrained StyleGAN2 mapping network** (`G.mapping`).

### How StyleGAN2's Mapping Network Works

In the original StyleGAN2, a single mapping network takes a 512-dim latent vector **z** and maps it to an intermediate latent vector **w** (also 512-dim). This **w** is then broadcast to all synthesis layers to form the **W** tensor of shape `(num_ws, 512)`. In our case, `num_ws = 14` for 256×256 resolution.

### Our Modification: 5 Separate Mappers

Instead of one mapping network, we use **5 independent mapping networks**. Each mapper:
1. **Input**: Takes the same 512-dim **CLIP embedding** of a real image
2. **Processing**: Passes it through the mapping network (8 fully-connected layers with LeakyReLU, as in the original StyleGAN2)
3. **Output**: Produces a 512-dim **w** vector
4. **Truncation**: Applies the truncation trick: $w' = \bar{w} + \psi \cdot (w - \bar{w})$ where $\bar{w}$ is the average latent from the pretrained model

### Layer Assignments

Each mapper's output **w** vector is assigned to specific synthesis layers:

```
Mapper 0 → w layers [0, 1]  → controls 4×4 resolution block   (coarse structure)
Mapper 1 → w layers [2, 3]  → controls 8×8 resolution block   (coarse structure)
Mapper 2 → w layers [4, 5]  → controls 16×16 resolution block (medium features)
Mapper 3 → w layers [6, 7]  → controls 32×32 resolution block (medium features)
Mapper 4 → w layers [8–13]  → controls 64×64, 128×128, 256×256 (fine details)
```

The same **w** from each mapper is broadcast to all its assigned layers (e.g., Mapper 0's output fills both layer 0 and layer 1). This gives us the complete **W+** tensor of shape `(B, 14, 512)` that the synthesis network expects.

### Why Deep Copy from Pretrained?

By starting from the pretrained mapping network weights, the mappers already understand the latent space W and can produce meaningful **w** vectors from the start. Training converges faster compared to randomly initialised networks.

In [15]:
class FiveMappers(nn.Module):
    """
    5 mapping networks, each deep-copied from the pretrained StyleGAN2 mapping network.
    Each takes a 512-dim CLIP embedding and outputs a 512-dim w vector.

    Layer assignments (for 256x256 resolution, num_ws=14):
        Mapper 0 → 4x4     (w layers 0, 1)
        Mapper 1 → 8x8     (w layers 2, 3)
        Mapper 2 → 16x16   (w layers 4, 5)
        Mapper 3 → 32x32   (w layers 6, 7)
        Mapper 4 → 64x64, 128x128, 256x256 (w layers 8–13)

    Truncation trick is applied: w' = w_avg + psi * (w - w_avg)
    """
    def __init__(self, pretrained_mapping, num_ws=14, w_dim=512, truncation_psi=0.7):
        super().__init__()
        self.num_ws = num_ws
        self.w_dim = w_dim
        self.truncation_psi = truncation_psi

        # Deep-copy pretrained mapping network 5 times
        self.mappers = nn.ModuleList([copy.deepcopy(pretrained_mapping) for _ in range(5)])

        # Enable gradient for all mapper parameters (they were frozen in the original G)
        for mapper in self.mappers:
            for p in mapper.parameters():
                p.requires_grad = True

        # Store w_avg from the pretrained model for truncation
        self.register_buffer('w_avg', pretrained_mapping.w_avg.clone())

        # Which synthesis layers each mapper feeds
        self.layer_assignments = [
            [0, 1],                      # Mapper 0 → 4×4
            [2, 3],                      # Mapper 1 → 8×8
            [4, 5],                      # Mapper 2 → 16×16
            [6, 7],                      # Mapper 3 → 32×32
            list(range(8, num_ws)),      # Mapper 4 → 64×64, 128×128, 256×256
        ]

    def forward(self, clip_embedding):
        """
        clip_embedding: (B, 512) float32 CLIP embedding
        Returns: ws (B, num_ws, w_dim) — the w+ tensor for the synthesis network
        """
        # Get one w vector per mapper
        w_vectors = []
        for mapper in self.mappers:
            # Use mapping network: z=clip_embedding, c=None (unconditional), no internal truncation
            w_broadcast = mapper(clip_embedding, c=None, truncation_psi=1.0)
            # w_broadcast: (B, num_ws, w_dim) — all layers identical
            w = w_broadcast[:, 0, :]  # (B, w_dim)

            # Truncation trick
            w = self.w_avg + self.truncation_psi * (w - self.w_avg)
            w_vectors.append(w)

        # Build w+ tensor: (B, num_ws, w_dim) by assigning each mapper's w to its target layers
        # Using torch.stack (no in-place ops) so autograd works cleanly
        layer_to_w = {}
        for mapper_idx, layers in enumerate(self.layer_assignments):
            for layer_idx in layers:
                layer_to_w[layer_idx] = w_vectors[mapper_idx]

        ws = torch.stack([layer_to_w[i] for i in range(self.num_ws)], dim=1)
        return ws

## Step 9 – Instantiate the Five Mappers

Read `num_ws` (14 for 256×256), `w_dim` (512), and `c_dim` (0 for unconditional FFHQ) from the loaded StyleGAN2 model. Then create the `FiveMappers` module:
- Deep-copies `G.mapping` five times
- Enables gradients on all mapper parameters (since G was frozen)
- Stores `w_avg` from the pretrained model for truncation
- Sets truncation factor $\psi = 0.7$

In [16]:
# Detect num_ws and w_dim from the pretrained model
num_ws = getattr(G.synthesis, 'num_ws', None) or getattr(G, 'num_ws', None) or 14
w_dim  = getattr(G, 'w_dim', 512)
c_dim  = getattr(G, 'c_dim', 0)

print(f'StyleGAN2 config: num_ws={num_ws}, w_dim={w_dim}, c_dim={c_dim}')

# Create 5 mapping networks, each initialised from the pretrained G.mapping
TRUNCATION_PSI = 0.7

mappers = FiveMappers(
    pretrained_mapping=G.mapping,
    num_ws=num_ws,
    w_dim=w_dim,
    truncation_psi=TRUNCATION_PSI,
).to(device)

total_params     = sum(p.numel() for p in mappers.parameters())
trainable_params = sum(p.numel() for p in mappers.parameters() if p.requires_grad)
print(f'Five mapping networks created from pretrained StyleGAN2 mapping network.')
print(f'Total params: {total_params:,} | Trainable params: {trainable_params:,}')

StyleGAN2 config: num_ws=14, w_dim=512, c_dim=0
Five mapping networks created from pretrained StyleGAN2 mapping network.
Total params: 10,506,240 | Trainable params: 10,506,240


## Step 10 - Dataset Split and DataLoaders

Load the full face image dataset, then split it into **train/validation/test** subsets before training starts.

1. Build one dataset from `FFHQ_DIR`
2. Split into train, val, and test using `VAL_SPLIT` and `TEST_SPLIT`
3. Train loader is used for optimization
4. Validation and test loaders are reserved for evaluation and final testing

This ensures the model is trained only on the training subset and does not see validation/test images during optimization.

In [ ]:
class SimpleImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.image_files = sorted([
            os.path.join(root, f)
            for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img = Image.open(self.image_files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

full_dataset = SimpleImageDataset(FFHQ_DIR, transform=real_image_transform)
dataset_size = len(full_dataset)

if dataset_size < 3:
    raise ValueError(f"Need at least 3 images for train/val/test split, found {dataset_size}.")

val_size = max(1, int(dataset_size * VAL_SPLIT))
test_size = max(1, int(dataset_size * TEST_SPLIT))
train_size = dataset_size - val_size - test_size

if train_size < 1:
    raise ValueError(
        f"Invalid split sizes for dataset size {dataset_size}. "
        f"Got train={train_size}, val={val_size}, test={test_size}. "
        "Reduce VAL_SPLIT/TEST_SPLIT or use more images."
    )

split_generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=split_generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
 )
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
 )
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
 )

print(f"Dataset loaded: total={dataset_size} | train={len(train_dataset)} | val={len(val_dataset)} | test={len(test_dataset)}")

Dataset loaded: 70000 images


## Step 11 – Training Loop

The training pipeline for each batch:

### Forward Pass
1. **Real images** → Frozen **CLIP ViT-B/32** → 512-dim embedding `clip_real` (detached, no grad)
2. `clip_real` → **5 Mapping Networks** (trainable) → W+ tensor of shape `(B, 14, 512)`
3. W+ → Frozen **Synthesis Network** (`G.synthesis`) → Generated image `(B, 3, 256, 256)`

### Loss Computation
4. Generated image → Frozen **Discriminator** → Adversarial loss: $\mathcal{L}_{adv} = \text{softplus}(-D(x_{fake}))$
5. Generated image → Frozen **CLIP** → `clip_fake` embedding → CLIP loss: $\mathcal{L}_{CLIP} = 1 - \cos(\text{clip\_real}, \text{clip\_fake})$
6. **Total loss**: $\mathcal{L} = \mathcal{L}_{adv} + \lambda \cdot \mathcal{L}_{CLIP}$

### Backward Pass
7. Gradients flow **only** through the 5 mapping networks (all other components are frozen)
8. Adam optimizer updates the mapper parameters

### Checkpointing
- Every 500 steps: save mapper weights and a sample image grid to Google Drive

In [18]:
optimizer = torch.optim.Adam(mappers.parameters(), lr=LR, betas=(0.9, 0.999))

step = 0
print_every = 50
save_every  = len(train_loader)
lambda_increment_per_x_epochs = 20
lamda_increment = 0.2

print(f'Training config: lr={LR}, lambda_clip={LAMBDA_CLIP}, truncation_psi={TRUNCATION_PSI}')
print(f'Frozen: G.synthesis, D, clip_model  |  Trainable: mappers (5 mapping networks)')

Training config: lr=0.002, lambda_clip=1, truncation_psi=0.7
Frozen: G.synthesis, D, clip_model  |  Trainable: mappers (5 mapping networks)


In [19]:
import wandb
import logging
from datetime import datetime
import os

RUN_NAME = f"clip_mapper_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
try:
    wandb.login()
except:
    pass
# ---- WandB init ----
run = wandb.init(
    project="clip-stylegan-mapper",
    name=RUN_NAME,
    config={
        "lr": LR,
        "lambda_clip_init": LAMBDA_CLIP,
        "truncation_psi": TRUNCATION_PSI,
        "epochs": EPOCHS,
        "print_every": print_every,
        "save_every": save_every
    }
)

# ---- File logger ----
LOG_FILE = os.path.join(SAVE_DIR, f"{BATCH_SIZE}-{LR}-{EPOCHS}-{LAMBDA_CLIP}-{RUN_NAME}.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        # logging.StreamHandler()  # still prints to notebook
    ]
)

logger = logging.getLogger()

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/m2s/.netrc.
wandb: Currently logged in as: saffrotech (saffrotech-uom) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
from tqdm import tqdm

for epoch in range(1, EPOCHS + 1):
    logger.info(f"Starting Epoch Number {epoch}")
    pbar = tqdm(total=len(train_loader), desc=f"Epoch {epoch}/{EPOCHS}", leave=True)

    epoch_train_loss_total = 0.0
    epoch_train_adv_total = 0.0
    epoch_train_clip_total = 0.0
    epoch_train_cos_total = 0.0
    num_train_batches = 0

    for batch_idx, real_imgs in enumerate(train_loader):
        mappers.train()
        optimizer.zero_grad()

        real_imgs = real_imgs.to(device)  # (B, 3, 256, 256) in [-1, 1]

        # ---- 1) CLIP-encode real images (frozen, no grad) ----
        clip_real = encode_with_clip_from_tensor(real_imgs, allow_grad=False).detach()
        # clip_real: (B, 512) float32

        # ---- 2) Map CLIP embeddings → w+ via the 5 mapping networks ----
        ws = mappers(clip_real)  # (B, num_ws, w_dim) float32

        # ---- 3) Synthesise images (frozen synthesis network) ----
        fake_images = G.synthesis(ws, noise_mode='const')
        # fake_images: (B, 3, 256, 256) in [-1, 1]

        # ---- 4) Discriminator loss (non-saturating G loss) ----
        try:
            d_fake = D(fake_images, c=torch.zeros(real_imgs.shape[0], c_dim, device=device))
        except TypeError:
            d_fake = D(fake_images)
        if isinstance(d_fake, (tuple, list)):
            d_fake = d_fake[0]
        g_adv_loss = F.softplus(-d_fake).mean()

        # ---- 5) CLIP similarity loss ----
        clip_fake = encode_with_clip_from_tensor(fake_images, allow_grad=True)
        # clip_fake: (B, 512) float32
        cos_sim   = F.cosine_similarity(clip_real, clip_fake, dim=-1)
        clip_diff = (1.0 - cos_sim).mean()

        # ---- 6) Total loss ----
        loss = g_adv_loss + LAMBDA_CLIP * clip_diff

        # ---- 7) Backward & update (only mapper params get gradients) ----
        loss.backward()
        optimizer.step()

        epoch_train_loss_total += loss.item()
        epoch_train_adv_total += g_adv_loss.item()
        epoch_train_clip_total += clip_diff.item()
        epoch_train_cos_total += cos_sim.mean().item()
        num_train_batches += 1

        # ---- Logging ----
        if step % print_every == 0:
            log_dict = {
                "loss_total": loss.item(),
                "loss_adv": g_adv_loss.item(),
                "loss_clip": clip_diff.item(),
                "cosine_similarity": cos_sim.mean().item(),
                "epoch": epoch,
                "step": step,
            }

            run.log(log_dict)

            logger.info(
                f"[Step {step}] Epoch {epoch} Batch {batch_idx} | "
                f"Loss: {loss.item():.4f} | Adv: {g_adv_loss.item():.4f} | "
                f"CLIP diff: {clip_diff.item():.4f} | Cos sim: {cos_sim.mean().item():.4f}"
            )

        # ---- Checkpointing ----
        if step % save_every == 0 and step > 0:
            ckpt_path = os.path.join(
                SAVE_DIR,
                f'step-{step}_lamda-{LAMBDA_CLIP}_lr-{LR}_batchz-{BATCH_SIZE}.pt'
            )
            torch.save({
                'mappers_state_dict': mappers.state_dict(),
                'optimizer':          optimizer.state_dict(),
                'step':               step,
                'epoch':              epoch,
                'truncation_psi':     TRUNCATION_PSI,
            }, ckpt_path)
            print(f'  → Saved checkpoint: {ckpt_path}')

            with torch.no_grad():
                sample_ws   = ws[:min(4, ws.shape[0])]
                sample_imgs = G.synthesis(sample_ws, noise_mode='const')
                grid = utils.make_grid((sample_imgs + 1) / 2, nrow=sample_imgs.shape[0])
                sample_path = os.path.join(SAVE_DIR, f'samples_step_{step}.png')
                utils.save_image(grid, sample_path)
                print(f'  → Saved samples:    {sample_path}')
            run.log({
                "sample_grid": wandb.Image(grid),
                "epoch": epoch,
                "lamda_clip_value": LAMBDA_CLIP
            })

        step += 1
        pbar.update(1)
        pbar.set_postfix({
            "loss": loss.item(),
            "adv": g_adv_loss.item(),
            "clip": clip_diff.item(),
            "cos": cos_sim.mean().item()
        })

    pbar.close()

    train_loss_epoch = epoch_train_loss_total / max(1, num_train_batches)
    train_adv_epoch = epoch_train_adv_total / max(1, num_train_batches)
    train_clip_epoch = epoch_train_clip_total / max(1, num_train_batches)
    train_cos_epoch = epoch_train_cos_total / max(1, num_train_batches)

    # ---- Validation pass (no gradients, no parameter updates) ----
    mappers.eval()
    val_loss_total = 0.0
    val_adv_total = 0.0
    val_clip_total = 0.0
    val_cos_total = 0.0
    num_val_batches = 0

    with torch.no_grad():
        for real_imgs in val_loader:
            real_imgs = real_imgs.to(device)

            clip_real = encode_with_clip_from_tensor(real_imgs, allow_grad=False).detach()
            ws = mappers(clip_real)
            fake_images = G.synthesis(ws, noise_mode='const')

            try:
                d_fake = D(fake_images, c=torch.zeros(real_imgs.shape[0], c_dim, device=device))
            except TypeError:
                d_fake = D(fake_images)
            if isinstance(d_fake, (tuple, list)):
                d_fake = d_fake[0]

            g_adv_loss = F.softplus(-d_fake).mean()
            clip_fake = encode_with_clip_from_tensor(fake_images, allow_grad=False)
            cos_sim = F.cosine_similarity(clip_real, clip_fake, dim=-1)
            clip_diff = (1.0 - cos_sim).mean()
            val_loss = g_adv_loss + LAMBDA_CLIP * clip_diff

            val_loss_total += val_loss.item()
            val_adv_total += g_adv_loss.item()
            val_clip_total += clip_diff.item()
            val_cos_total += cos_sim.mean().item()
            num_val_batches += 1

    val_loss_epoch = val_loss_total / max(1, num_val_batches)
    val_adv_epoch = val_adv_total / max(1, num_val_batches)
    val_clip_epoch = val_clip_total / max(1, num_val_batches)
    val_cos_epoch = val_cos_total / max(1, num_val_batches)

    run.log({
        "train_loss_epoch": train_loss_epoch,
        "train_adv_epoch": train_adv_epoch,
        "train_clip_epoch": train_clip_epoch,
        "train_cosine_similarity_epoch": train_cos_epoch,
        "val_loss_epoch": val_loss_epoch,
        "val_adv_epoch": val_adv_epoch,
        "val_clip_epoch": val_clip_epoch,
        "val_cosine_similarity_epoch": val_cos_epoch,
        "epoch": epoch,
        "step": step,
        "lamda_clip_value": LAMBDA_CLIP,
    })

    logger.info(
        f"[Epoch {epoch}] TrainLoss: {train_loss_epoch:.4f} | ValLoss: {val_loss_epoch:.4f} | "
        f"TrainAdv: {train_adv_epoch:.4f} | ValAdv: {val_adv_epoch:.4f} | "
        f"TrainCLIP: {train_clip_epoch:.4f} | ValCLIP: {val_clip_epoch:.4f} | "
        f"TrainCos: {train_cos_epoch:.4f} | ValCos: {val_cos_epoch:.4f}"
    )

    if epoch % lambda_increment_per_x_epochs == 0 and epoch > 0:
        LAMBDA_CLIP += lamda_increment
print('Training finished.')

Epoch 2/100:   0%|                                                                             | 0/4375 [00:00<?, ?it/s]wandb: WARNING Data passed to `wandb.Image` should consist of values in the range [0, 255], image data will be normalized to this range, but behavior will be removed in a future version of wandb.


  → Saved checkpoint: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/5_mapper_b16/step-4375_lamda-1_lr-0.002_batchz-16.pt
  → Saved samples:    /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/5_mapper_b16/samples_step_4375.png


Epoch 3/100:   0%|                     | 1/4375 [00:00<34:35,  2.11it/s, loss=0.271, adv=0.00918, clip=0.262, cos=0.738]

  → Saved checkpoint: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/5_mapper_b16/step-8750_lamda-1_lr-0.002_batchz-16.pt
  → Saved samples:    /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/5_mapper_b16/samples_step_8750.png


Epoch 4/100:   0%|                       | 1/4375 [00:00<32:07,  2.27it/s, loss=0.198, adv=0.006, clip=0.192, cos=0.808]

...

Epoch 28/100:  36%|██████           | 1568/4375 [05:50<10:27,  4.47it/s, loss=0.111, adv=0.00342, clip=0.0901, cos=0.91]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Epoch 53/100:  75%|███████████▉    | 3273/4375 [12:12<04:06,  4.47it/s, loss=0.103, adv=0.00303, clip=0.0712, cos=0.929]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Epoch 59/100:   0%|                  | 1/4375 [00:00<32:59,  2.21it/s, loss=0.0814, adv=0.00312, clip=0.0559, cos=0.944]

  → Saved checkpoint: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/5_mapper_b16/step-253750_lamda-1.4_lr-0.002_batchz-16.pt
  → Saved samples:    /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/5_mapper_b16/samples_step_253750.png


Epoch 60/100:   0%|                  | 1/4375 [00:00<32:29,  2.24it/s, loss=0.0919, adv=0.00312, clip=0.0634, cos=0.937]

  → Saved checkpoint: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/5_mapper_b16/step-258125_lamda-1.4_lr-0.002_batchz-16.pt
  → Saved samples:    /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/5_mapper_b16/samples_step_258125.png


Epoch 60/100:   4%|▌               | 157/4375 [00:35<15:43,  4.47it/s, loss=0.0825, adv=0.00325, clip=0.0566, cos=0.943]

In [ ]:
run.finish()